In [ ]:
# NCrystal notebook settings
# title: The NCrystal MiniMC simulation engine
# shortkey: minimc
# section: basics
# requires: plot

NCrystal ships with a builtin stepping Monte Carlo, which allows to model effects of multiple scattering and geometry as neutrons are going through a simple geometry consisting of a single convex volume (box, sphere, cylinder, or slab) filled with a single homogenous material. In this notebook we will give a few basic examples of how this might be used via the Python API. Note that on the commandline, the `ncrystal_minimc` command also provides access to the MiniMC engine. Run `ncrystal_minimc --help` for more information.

# Introduction
First import NCrystal and the minimc module:

In [ ]:
import NCrystal as NC
assert NC.version_tuple >= (4,2,13), "too old NCrystal version"
import NCrystal.minimc as mmc

We shall do a few quick demonstrations of how the MiniMC can be used. For reference, note that there are several ways to access more information about this. For instance one might visit the documentation at https://github.com/mctools/ncrystal/wiki/minimc. But documentation is also available inline. For reference, we note that one might uncomment and run any of the following statements to access documentation directly here in this notebook:

In [ ]:
#help(mmc)
#mmc.gen_doc('src')
#mmc.gen_doc('geom')
#mmc.gen_doc('engine')
#mmc.gen_doc('scenario')

# Basic usage
Here we will be running the MiniMC engine with an NCrystal cfg-string, here representing a Zinc Oxide powder at room temperature:

In [ ]:
mmc.run('ZnO_sg186_ZincOxide.ncmat;temp=300K')

So what happened? Well, we ran a MiniMC simulation and got the results back in an MMCResults object. We can also see that although we didn't specify any particular neutron source or material geometry, NCrystal selected something which was hopefully "interesting". In this case, what was chosen was to send a uniform circular beam of ~4.5Aa neutrons through a ~2cm radius sphere. For actual scientific work, one will of course almost always want to perform a more careful configuration, and we will show how to do that later.

For now, let us run the simulation again and this time capture the results in a variable so we can investigate them a bit. As an example, we will plot the default tally, which is the scattering angle of the neutrons. Note that neutrons are tallied as they leave the geometry (in this case a sphere filled with our zinc oxide):

In [ ]:
results_ZnO = mmc.run('ZnO_sg186_ZincOxide.ncmat;temp=300K')
results_ZnO.plot()

Here above we see 3 distinct Bragg peaks due to neutrons undergoing a single elastic scattering, and in red the contribution of neutrons undergoing a single inelastic scattering. In light-blue are shown several spurious "peaks" which are due to neutrons undergoing multiple elastic scatterings on the 3 primary peaks. Finally, in orange is the contribution from multiple scattering events which involves at least one inelastic scattering.

The above histogram depends on the parameters of the incoming (source) neutrons, the simulation geometry, and of course the NCrystal model of the neutron interactions in the chosen material. To investigate the latter, one could also feed the material cfg-string to the `plot_xsect` function from the `NCrystal.plot` module. For convenience, the MiniMC results object contains a `plot_xsect` method which does just that:

In [ ]:
results_ZnO.plot_xsect()

## Specifying source and geometry
Now, let us modify the source and geometry. To do so, one can supply two configuration string: one for the neutron source (`srccfg`), and on for the geometry (`geomcfg`). Alternatively, one can for convenience use a single `scenario` configuration string. For instance, one can simulate a scenario in which a pencil beam of 1e5 neutrons of wavelength 5.5Aa impinge upon a sphere of diameter 2cm by running:

In [ ]:
results_ZnO_v2 = mmc.run('ZnO_sg186_ZincOxide.ncmat;temp=300K',
                         scenario='5.5Aa pencil on 2cm sphere 1e5 times')

We can of course plot the results and see the change:

In [ ]:
results_ZnO_v2.plot()

But we can also verify by simply inspecting the setup actually used, by looking in the dictionary available through the `.setup` property:

In [ ]:
print(results_ZnO_v2.setup['src']['cfgstr'])
print(results_ZnO_v2.setup['geom']['cfgstr'])

The scenario cfg-strings are mostly intended for quick usage (for instance at the cmd-line when using the `ncrystal_minimc` command). For more information visit https://github.com/mctools/ncrystal/wiki/minimc_scenario or uncomment and run:

In [ ]:
#mmc.gen_doc('scenario')

## Accessing results
Now, let us dive a bit more into how one can access and use MiniMC results. First of all, note that in addition to histograms of tallied quantities, overall statistics count is available on the `.output_metadata` property, which lets us know how many (`count`) and with what flux (`weight`) neutrons were generated by the source, missed the geometry, or were available for the tally. Note, that neutrons absorbed inside the geometry will reduce the tallied flux. Also note that variance-reduction tricks like splitting of particle histories are used, so the number (`count`) of neutrons tallied will usually be higher than the number of neutrons emitted from the source:

In [ ]:
results_ZnO_v2.output_metadata

In addition to merely plotting tally histograms, we can of course also access them programmatically as the following code examples show:

In [ ]:
print("Available tallies:")
print(results_ZnO_v2.tally_names)
t_theta = results_ZnO_v2.tally('theta')
print(type(t_theta))
#Uncomment for documentation:
#help( t_theta) )

We can also get a bit of information by the `.dump(..)` method:

In [ ]:
t_theta.dump()

And we can access histograms as well, either one of the components (like `SINGLESCAT_ELAS`) or the `total`. This then allows us to access bin contents:

In [ ]:
t_theta.histograms

In [ ]:
import pprint
h_sglelas = t_theta.histograms['SINGLESCAT_ELAS']
print( h_sglelas.binning )
print( h_sglelas.contents )
print( h_sglelas.errors )

## Configuring tallies
Next, assume we want to tally something else than the scattering angle. For instance, the energy loss (`de`). For that we have to use the `enginecfg` string, and add the quantities we want to tally as a comma separated list using the `tally` keyword. For this example, let us tally `de` in addition to the scattering angle (`theta`): 

In [ ]:
results_ZnO_v3 = mmc.run('ZnO_sg186_ZincOxide.ncmat;temp=300K',
                         scenario='5.5Aa pencil on 2cm sphere 1e5 times',
                         enginecfg='tally=de,theta')
#Uncomment next line to plot both "de" and "theta" histograms:
#results_ZnO_v3.plot()#
#Here is how to plot just "de":
results_ZnO_v3.tally('de').plot()

In the plot above, we might not be happy with the chosen histogram binning. We can use the `tallybins` parameter to override it:

In [ ]:
results_ZnO_v4 = mmc.run('ZnO_sg186_ZincOxide.ncmat;temp=300K',
                         scenario='5.5Aa pencil on 2cm sphere 1e5 times',
                         enginecfg='tally=de,theta;tallybins=de:200:-0.25:0.05')
#Uncomment next line to plot both "de" and "theta histograms:
#results_ZnO_v3.plot()
#Here is how to plot just "de":
results_ZnO_v4.tally('de').plot()

There are many more things that can be tuned in the enginecfg string. Access more information about it by uncommenting and running the next line:

In [ ]:
#mmc.gen_doc('engine')

Likewise, for more advanced usage one will typically want to specify `srccfg` and `geomcfg` strings rather than just the scenario strings we saw above. For more information about this, uncomment and run the lines below:

In [ ]:
#mmc.gen_doc('src')
#mmc.gen_doc('geom')

# Advanced subjects

## Saving results
Under the hood the output of a MiniMC run is actually a JSON string containing a dictionary with all the information related to the run (both configuration data and output). The `MMCResults` class that we have been manipulating so far, is actually just a thin convenience wrapper around this dictionary. For that reason, results can be easily saved to a file and loaded at a later point. This might be useful if a large-statistics run required a significant running time. As an example, here is how one converts a results object to a JSON string:

In [ ]:
my_json_data = results_ZnO.to_json()
print( type(my_json_data), len(my_json_data) )

So around 40kB of data in this case. Here is how it is converted back and used:

In [ ]:
results_reloaded = mmc.MMCResults(my_json_data)
results_reloaded.plot()

If desired, the JSON data can be saved and loaded to a file using the `pathlib` module's `Path.write_text` and `Path.read_text` methods. For instance:

In [ ]:
import pathlib
#Store the data:
pathlib.Path('./my_minimc_results.json').write_text(my_json_data)
#Load the data again later:
results = mmc.MMCResults( pathlib.Path('./my_minimc_results.json').read_text() )

## Using a callback function

In case the predefined tally quantities are not enough, it is also possible to simply get access to all the tallied neutron data. This is done by defining a custom callback function which will be called by the MiniMC engine while the simulation is running. For full efficiency, neutron states are provided as numpy arrays, allowing large-scale processing in Python code in an efficient fashion. Let us show how this works by the following example in which we will print all the received data. To not overflow this notebook, we will run with a very low number of neutrons:

In [ ]:
import pprint
def mycallbackfct( data ):
    print("I got some data!")
    pprint.pp(data)
results_callback = mmc.run('Be_sg194.ncmat',
                           scenario='3.8Aa on 1cm sphere 2 times',
                           callback = mycallbackfct)
results_callback.plot()

Several points can already be made based on this little example.

First of all, we still got the usual results object back via the `mmc.run(..)` return value, and could use it to plot the scattering angle tally just as we did previously. Thus, the presence or absence of a callback function does not otherwise alter how one configures a simulation or what results are returned. It merely gives us a way to spy on the neutrons via the callback function.

We saw how the callback function gets the data as a dictionary where the keys are names of neutron state parameters ('x','y','z','ekin',...) and the values are 1D Numpy arrays with the values. All the 1D arrays have the same length, and this length is the number of neutrons in that callback.

We didn't see it in this low statistics run, but the callback function will usually get invoked more than once. Let us try to see this in an example with more normal statistics:

In [ ]:
def mycallbackfct( data ):
    print("mycallbackfct: I got %i neutrons"%len(data['ekin']))
results_callback = mmc.run('Be_sg194.ncmat',
                           scenario='3.8Aa on 1cm sphere 1e6 times',
                           callback = mycallbackfct)

So we see we got called several times, everytime (except possibly the last) with a very large number of neutrons (to have arrays long enough to treat efficiently in Python). The reason we do not simply provide a single call to the callback function with all the neutrons, is of course that this could result in too large memory requirements.

Now let us do something a bit more useful. Assume we want to obtain the energy distribution of neutrons transmitted through a 1cm slab of polyethylene and headed within 10degrees of the forward direction. This particular criteria can not be accessed via one of the standard tallies, but can be found via the callback function.

Naturally, it is up to the user how to collect and analyse the data passed to the callback function, but here we will create histograms with the help of a utility class `HistFiller1D` from NCrystal.

In [ ]:
from NCrystal.hist import HistFiller1D
import math
h = HistFiller1D( 100, 0.0, 0.1, "E (eV) for fwd emitted neutrons")
def mycallbackfct( data ):
    mask = data['uz'] > math.cos(10.0*math.pi/180)
    n, nused = len(data['ekin']), mask.sum()
    print("mycallbackfct: I got %i neutrons (using %.1f%%)"%(n,nused*100.0/n))
    h.fill( data['ekin'][mask], data['w'][mask] )
results_callback = mmc.run('Polyethylene_CH2.ncmat;temp=100K',
                           scenario='25meV on 1cm slab 1e5 times',
                           callback = mycallbackfct)
#to plot our custom histogram we convert the HistFiller1D object
#to a Hist1D object via .to_hist1D() and then call plot:
plt = h.to_hist1d().plot(logy=True)

So this is how a callback function could be used. Notice btw. that the fraction of neutrons fulfilling our directional `fwd` criteria of `uz>cos(10degree)` is not constant when we are getting callbacks. This is due to our neutron splitting and variance reduction schemes. Thus, to avoid bias it is important to always let the run finish and include neutrons from all callback invocations equally. It *is* however possible to let the run terminate dynamically once "enough" statistics has been acquired. Simply return `True` from the callback function, and the neutron source will instantely "dry up" even though the originally requested number of source neutrons has not been delivered. Any lingering neutrons already in the system will however be allowed to carry on with their scatterings and be tallied as usual. This is intended to avoid any sort of bias which might otherwise result from the simulation being aborted immediately.

## Command-line interface
For completeness, we mention here that it is also possible to launch the MiniMC directly from a terminal interface as well. This is done using the `ncrystal_minimc` command.

To try it out, please open a terminal and type the command:
```
ncrystal_minimc "Al_sg225.ncmat" "4Aa on 1cm sphere"`.
```
If you get an error like `Command 'ncrystal_minimc' not found`, it might be because you have to first install (or update) NCrystal, or perhaps activate a Python VENV or a CONDA environment in which NCrystal has been installed.

To include a quick example here in this notebook, we will use the `--dump` option to get textual output rather than a plot, since the opening of a graphics window from the command-line might not work in all notebooks:

In [ ]:
!ncrystal_minimc --dump "Al_sg225.ncmat" "4Aa on 1cm sphere 1e5 times"

To get more detailed usage information, run the command:
```
ncrystal_minimc --help
```
Or for even more information:
```
ncrystal_minimc --full-help
```
This can of course also be done from this notebook, and this is how it looks:

In [ ]:
!ncrystal_minimc --help

# Reference documentation

To get more information about the NCrystal MiniMC, refer to the wiki at:
https://github.com/mctools/ncrystal/wiki/minimc

Alternatively, you can as already mentioned also access information directly from Python (like here in this notebook), by uncommenting and running commands like for instance `mmc.gen_doc(<subject>)`. For convenience, this is done in the following subsections.

## Docs: source cfg

In [ ]:
mmc.gen_doc('src')

## Docs: geometry cfg

In [ ]:
mmc.gen_doc('geom')

## Docs: scenario strings (source+geometry)

In [ ]:
mmc.gen_doc('scenario')

## Docs: engine/tally cfg

In [ ]:
mmc.gen_doc('engine')

## Docs: builtin Python help
Like the rest of the NCrystal Python API, the classes and functions related to the MiniMC also have built-in information, which can for instance be accessed via the Python `help()` command. Here is an example:

In [ ]:
help(mmc.run)

## Docs: command-line interface

In [ ]:
!ncrystal_minimc --full-help